# Imports

In [ ]:
import dataclasses
import json
import shutil
from pathlib import Path

import numpy as np
import onnxruntime as ort
import pandas as pd
import torch
from tqdm.auto import tqdm

from uavloc.data.drone_frames import load_frames
from uavloc.data.multidate import load_date
from uavloc.data.satellite import GeoImage
from uavloc.features.factory import correlator_config
from uavloc.features.segmentation import SegmentationFeatures, load_segmenter
from uavloc.geo import LatLon, haversine_m
from uavloc.matching.correlator import PositionCorrelator
from uavloc.track import REPO_ROOT, experiment_dir

# Params

In [ ]:
EXP = "021-building-loss-from-scratch"
ARCHITECTURES = ["unet-resnet18", "unet-resnet34", "lraspp-mobilenetv3"]
LOSS = "current"
TRAIN_SEED = 42
STEPS = 2500
RULE = "gated"                                   # decision rule of the case, from 021 correction-config.json
NOISE_M = 10.0
BENCH_SHAPES = {"drone": [1, 3, 416, 736], "satellite": [1, 3, 512, 512]}   # same inputs as pocs/lraspp-pi5
DYNAMIC_CHECK_SHAPE = [1, 3, 96, 128]            # a correction-sized input, checks the dynamic ONNX axes
OPSET = 17
SEED = 0
MAP_HALF_SIZE_M = 100.0                          # satellite window shipped to the Pi, around the prior
SAT_DEVICE = "mps"
OUT_DIR = REPO_ROOT / "pocs" / "seg-correction-pi5"

# Data

In [ ]:
ARTIFACTS = experiment_dir(EXP, create=False) / "artifacts"
MODEL_DIR, CASE_DIR = Path(OUT_DIR) / "models", Path(OUT_DIR) / "case"
DRAW_KEYS = ["frame", "place", "noise_m", "bearing_deg", "prior_lat", "prior_lon", "prior_error_m"]
config = json.loads((ARTIFACTS / "correction-config.json").read_text())[RULE]
correlator_cfg = correlator_config(config)
correlator_cfg

In [ ]:
def arch_draws(arch: str) -> pd.DataFrame:
    """021 draws at NOISE_M under RULE for one architecture, error and decision suffixed with its name."""
    r = pd.read_csv(ARTIFACTS / f"correction-records-{arch}-{LOSS}-seed{TRAIN_SEED}.csv")
    r = r[(r["rule"] == RULE) & (r["noise_m"] == NOISE_M)]
    return r[DRAW_KEYS + ["error_m", "decision"]].rename(
        columns={"error_m": f"error_m_{arch}", "decision": f"decision_{arch}"})

In [ ]:
draws = arch_draws(ARCHITECTURES[0])
for arch in ARCHITECTURES[1:]:
    draws = draws.merge(arch_draws(arch), on=DRAW_KEYS)
accepted = draws[(draws[[f"decision_{a}" for a in ARCHITECTURES]] == "accept").all(axis=1)]
case = accepted.loc[accepted[[f"error_m_{a}" for a in ARCHITECTURES]].max(axis=1).idxmin()]
case

In [ ]:
frame = next(f for f in load_frames(config["frames_dataset"]) if f.name == case["frame"])
prior = LatLon(case["prior_lat"], case["prior_lon"])
satellite = load_date(case["place"], config["map_date"])
frame, haversine_m(prior, frame.position), satellite.pixels.shape

# Model

In [ ]:
def checkpoint_path(arch: str) -> Path:
    return ARTIFACTS / f"model-{arch}-{LOSS}-seed{TRAIN_SEED}-steps{STEPS}.pt"

In [ ]:
def export_onnx(model: torch.nn.Module, path: Path) -> None:
    """Batch 1, dynamic height and width: one file serves both bench shapes and the correction frame."""
    x = torch.randn(*BENCH_SHAPES["satellite"])
    axes = {2: "height", 3: "width"}
    torch.onnx.export(model, x, str(path), input_names=["image"], output_names=["logits"], opset_version=OPSET,
                      dynamic_axes={"image": axes, "logits": axes})

In [ ]:
@torch.no_grad()
def reference_io(model: torch.nn.Module, shape: list[int]) -> dict[str, np.ndarray]:
    """Seeded random input and the Mac torch CPU logits for it."""
    x = torch.randn(*shape, generator=torch.Generator().manual_seed(SEED))
    return {"x": x.numpy(), "logits": model(x).numpy()}

In [ ]:
def onnx_max_abs_diff(path: Path, io: dict[str, np.ndarray]) -> float:
    session = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    return float(np.abs(session.run(None, {"image": io["x"]})[0] - io["logits"]).max())

In [ ]:
def export_model(arch: str) -> dict:
    """ONNX + state dict + bench reference I/O for one architecture, with ONNX-vs-torch parity per shape."""
    model = load_segmenter(checkpoint_path(arch), "cpu", arch)
    onnx_path = MODEL_DIR / f"{arch}.onnx"
    export_onnx(model, onnx_path)
    shutil.copy(checkpoint_path(arch), MODEL_DIR / f"{arch}.pt")
    parity = {}
    for name, shape in {**BENCH_SHAPES, "dynamic-check": DYNAMIC_CHECK_SHAPE}.items():
        io = reference_io(model, shape)
        if name in BENCH_SHAPES:
            np.savez(MODEL_DIR / f"reference-{arch}-{name}.npz", **io)
        parity[name] = onnx_max_abs_diff(onnx_path, io)
    return {"architecture": arch, "checkpoint": checkpoint_path(arch).name,
            "params_m": sum(p.numel() for p in model.parameters()) / 1e6, **parity}

In [ ]:
def geo_window(image: GeoImage, center: LatLon, half_size_m: float) -> GeoImage:
    """North-up crop of about `2 * half_size_m` around `center`, same pixels and lat/lon mapping."""
    cx, cy = image.to_px(center)
    mpp_x, mpp_y = image.mpp_xy
    h, w = image.pixels.shape[:2]
    left, right = max(0, round(cx - half_size_m / mpp_x)), min(w, round(cx + half_size_m / mpp_x))
    top, bottom = max(0, round(cy - half_size_m / mpp_y)), min(h, round(cy + half_size_m / mpp_y))
    return GeoImage(np.ascontiguousarray(image.pixels[top:bottom, left:right]),
                    image.to_latlon(left, top), image.to_latlon(right, bottom))

In [ ]:
def satellite_probabilities(arch: str) -> GeoImage:
    """Whole-map class probabilities at the match GSD (the pre-flight step), cut to the window the Pi needs."""
    model = load_segmenter(checkpoint_path(arch), SAT_DEVICE, arch)
    features = SegmentationFeatures(model, model, satellite, config["seg_channels"], config["drone_seg_gsd_m"],
                                    config["match_gsd_m"], SAT_DEVICE)
    return geo_window(features.probabilities, prior, MAP_HALF_SIZE_M)

In [ ]:
def save_probabilities(probabilities: GeoImage, path: Path) -> None:
    np.savez(path, pixels=probabilities.pixels.astype(np.float32), top_left=np.array(probabilities.top_left),
             bottom_right=np.array(probabilities.bottom_right))

In [ ]:
def reference_correction(arch: str, probabilities: GeoImage) -> dict:
    """The correction on Mac CPU from the inputs the Pi gets; the Pi's result is checked against it."""
    features = SegmentationFeatures.from_satellite_probabilities(
        load_segmenter(checkpoint_path(arch), "cpu", arch), probabilities, config["seg_channels"],
        config["drone_seg_gsd_m"], config["match_gsd_m"], "cpu")
    correction = PositionCorrelator(features, correlator_cfg).correct(frame, prior)
    return {"corrected_lat": correction.position.lat, "corrected_lon": correction.position.lon,
            "error_m": haversine_m(correction.position, frame.position), "decision": correction.decision,
            "peak_z": correction.peak_z, "records_error_m": float(case[f"error_m_{arch}"])}

# Run

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CASE_DIR.mkdir(parents=True, exist_ok=True)
exports = pd.DataFrame([export_model(arch) for arch in tqdm(ARCHITECTURES, desc="export")]).set_index("architecture")
exports

In [ ]:
(MODEL_DIR / "export.json").write_text(json.dumps(
    {"opset": OPSET, "torch": torch.__version__, "onnxruntime": ort.__version__, "bench_shapes": BENCH_SHAPES,
     "exports": exports.reset_index().to_dict("records")}, indent=2))

In [ ]:
probabilities = {arch: satellite_probabilities(arch) for arch in tqdm(ARCHITECTURES, desc="satellite")}
for arch, p in probabilities.items():
    save_probabilities(p, CASE_DIR / f"satellite-{arch}.npz")
{arch: p.pixels.shape for arch, p in probabilities.items()}

In [ ]:
references = {arch: reference_correction(arch, probabilities[arch]) for arch in ARCHITECTURES}
pd.DataFrame(references).T

In [ ]:
shutil.copy(frame.image_path, CASE_DIR / "frame.jpg")
(CASE_DIR / "case.json").write_text(json.dumps({
    "experiment": EXP, "loss": LOSS, "rule": RULE, "place": case["place"], "noise_m": NOISE_M,
    "bearing_deg": float(case["bearing_deg"]), "prior": list(prior),
    "frame": {"name": frame.name, "position": list(frame.position), "altitude_m": frame.altitude_m,
              "yaw_deg": frame.yaw_deg, "mpp": frame.mpp},
    "seg_channels": config["seg_channels"], "drone_seg_gsd_m": config["drone_seg_gsd_m"],
    "match_gsd_m": config["match_gsd_m"], "correlator_config": dataclasses.asdict(correlator_cfg),
    "references": references}, indent=2))
sorted(p.name for p in CASE_DIR.iterdir())